In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")

In [5]:
df.shape

(541909, 8)

In [6]:
df["Revenue"] = df["Quantity"]*df["UnitPrice"]

In [7]:
df = df.drop_duplicates().copy()

In [ ]:
df = df[df["UnitPrice"]>=0].copy()
#df[...] (Filtreleme): Köşeli parantez içine bu True/False serisini verdiğinizde Pandas sadece True olan satırları seçer, False olanları (yani negatif fiyatları) eler.

In [9]:
df.shape

(536639, 9)

In [10]:
product_sales = (
    df[df["Quantity"] > 0]
    .groupby("StockCode")["Quantity"]
    .sum() 
)

In [11]:
product_sales.sort_values(ascending=False).head(10) 

StockCode
23843     80995
23166     78033
22197     56898
84077     54951
85099B    48375
85123A    41645
21212     36396
84879     36362
23084     31673
22492     26633
Name: Quantity, dtype: int64

In [16]:
top_10_products = product_sales.sort_values(ascending=False).head(10)

In [17]:
product_info = df[["StockCode", "Description"]].drop_duplicates("StockCode")

In [18]:
top_10_products = (
    top_10_products
    .reset_index()
    .merge(product_info, on="StockCode", how = "left")
)

In [19]:
top_10_products

,StockCode,Quantity,Description
0,23843,80995,"PAPER CRAFT , LITTLE BIRDIE"
1,23166,78033,MEDIUM CERAMIC TOP STORAGE JAR
2,22197,56898,SMALL POPCORN HOLDER
3,84077,54951,WORLD WAR 2 GLIDERS ASSTD DESIGNS
4,85099B,48375,JUMBO BAG RED RETROSPOT
5,85123A,41645,WHITE HANGING HEART T-LIGHT HOLDER
6,21212,36396,PACK OF 72 RETROSPOT CAKE CASES
7,84879,36362,ASSORTED COLOUR BIRD ORNAMENT
8,23084,31673,RABBIT NIGHT LIGHT
9,22492,26633,MINI PAINT SET VINTAGE


In [20]:
product_revenue = (
    df[df["Quantity"]>0]
    .groupby("StockCode")["Revenue"]
    .sum()
)

In [21]:
product_revenue.sort_values(ascending=False).head(10)

StockCode
DOT       206248.77
22423     174156.54
23843     168469.60
85123A    104462.75
47566      99445.23
85099B     94159.81
23166      81700.92
POST       78101.88
M          77750.27
23084      66870.03
Name: Revenue, dtype: float64

In [22]:
df[df["StockCode"].isin(["DOT","POST","M"])][
    ["StockCode","Description","Quantity","UnitPrice","Revenue"]
].head(20)

,StockCode,Description,Quantity,UnitPrice,Revenue
45,POST,POSTAGE,3,18.00,54.00
386,POST,POSTAGE,1,15.00,15.00
1123,POST,POSTAGE,1,18.00,18.00
1814,DOT,DOTCOM POSTAGE,1,569.77,569.77
2239,M,Manual,1,1.25,1.25
2250,M,Manual,1,18.95,18.95
3041,DOT,DOTCOM POSTAGE,1,607.49,607.49
5073,POST,POSTAGE,1,18.00,18.00
5258,POST,POSTAGE,1,18.00,18.00
5325,POST,POSTAGE,2,40.00,80.00


In [23]:
special_codes = ["DOT","POST","M"]

In [24]:
product_revenue_clean = (
    df[
        (df["Quantity"]>0)&
        (~df["StockCode"].isin(special_codes))
    ]
    .groupby("StockCode")["Revenue"]
    .sum()
)

In [25]:
product_revenue_clean.sort_values(ascending=False).head(10)

StockCode
22423     174156.54
23843     168469.60
85123A    104462.75
47566      99445.23
85099B     94159.81
23166      81700.92
23084      66870.03
22086      64875.59
84879      58927.62
79321      54096.36
Name: Revenue, dtype: float64

In [26]:
top_revenue_products = product_revenue_clean.sort_values(
    ascending=False
).head(10)

In [27]:
top_revenue_products = (
    top_revenue_products
    .reset_index()
    .merge(product_info, on ="StockCode", how ="left")
)

In [28]:
top_revenue_products

,StockCode,Revenue,Description
0,22423,174156.54,REGENCY CAKESTAND 3 TIER
1,23843,168469.60,"PAPER CRAFT , LITTLE BIRDIE"
2,85123A,104462.75,WHITE HANGING HEART T-LIGHT HOLDER
3,47566,99445.23,PARTY BUNTING
4,85099B,94159.81,JUMBO BAG RED RETROSPOT
5,23166,81700.92,MEDIUM CERAMIC TOP STORAGE JAR
6,23084,66870.03,RABBIT NIGHT LIGHT
7,22086,64875.59,PAPER CHAIN KIT 50'S CHRISTMAS
8,84879,58927.62,ASSORTED COLOUR BIRD ORNAMENT
9,79321,54096.36,CHILLI LIGHTS


In [29]:
df["InvoiceDate"].head()

0   2010-12-01 08:26:00
1   2010-12-01 08:26:00
2   2010-12-01 08:26:00
3   2010-12-01 08:26:00
4   2010-12-01 08:26:00
Name: InvoiceDate, dtype: datetime64[us]

In [30]:
df["InvoiceDate"].dtype

dtype('<M8[us]')

In [31]:
df["Hour"] = df["InvoiceDate"].dt.hour

In [32]:
hourly_orders = df["Hour"].value_counts().sort_index()

In [33]:
hourly_orders

Hour
6        41
7       383
8      8906
9     34314
10    48808
11    56949
12    77573
13    71247
14    66570
15    76938
16    54134
17    28371
18     7941
19     3617
20      847
Name: count, dtype: int64

In [34]:
df["InvoiceNo"].nunique()

25898

In [35]:
orders_by_hour = (
    df[["InvoiceNo", "Hour"]]
    .drop_duplicates()
    .groupby("Hour")["InvoiceNo"]
    .count()
)

In [36]:
orders_by_hour

Hour
6       22
7       31
8      624
9     1824
10    2961
11    3165
12    3962
13    3369
14    3135
15    3069
16    1952
17    1205
18     333
19     219
20      28
Name: InvoiceNo, dtype: int64

In [37]:
country_revenue = (
    df[df["Quantity"]>0]
    .groupby("Country")["Revenue"]
    .sum()
)

In [38]:
country_revenue.sort_values(ascending=False).head(10)

Country
United Kingdom    9001744.094
Netherlands        285446.340
EIRE               283140.520
Germany            228678.400
France             209625.370
Australia          138453.810
Spain               61558.560
Switzerland         57067.600
Belgium             41196.340
Sweden              38367.830
Name: Revenue, dtype: float64

In [39]:
customer_revenue = (
    df[
        (df["Quantity"]>0)&
        (df["CustomerID"].notna())
    ]
    .groupby("CustomerID")["Revenue"]
    .sum()
)

In [40]:
customer_revenue.sort_values(ascending=False).head(10)

CustomerID
14646.0    280206.02
18102.0    259657.30
17450.0    194390.79
16446.0    168472.50
14911.0    143711.17
12415.0    124914.53
14156.0    117210.08
17511.0     91062.38
16029.0     80850.84
12346.0     77183.60
Name: Revenue, dtype: float64